# Sprint 4 — Deployment Preparation

## Sprint Goal

Deploy the trained cardiovascular disease prediction model as a live, public application that can provide predictions for new users.

## Deployment Backlog

The Sprint 4 backlog includes:

1. Serialize the trained neural network model.
2. Serialize all preprocessing objects required for inference.
3. Verify that the serialized artifacts reproduce known predictions.
4. Freeze a clean, pinned `requirements.txt`.
5. Prepare the model inference pipeline.
6. Build the prediction API.
7. Build the user interface.
8. Containerize the application.
9. Deploy the application publicly.
10. Polish the application and documentation.

## Sprint 3 Retrospective Carry-Forward

The Sprint 3 improvements are carried forward by preserving the same feature engineering, preprocessing pipeline, trained model configuration, and optimized classification threshold during deployment.

The deployment pipeline must use the same preprocessing applied during training to prevent training-serving skew.

In [19]:
import os
import random
import sys
import numpy as np
import tensorflow as tf
import sklearn

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f"Seed: {SEED}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")

Seed: 42
Artifacts directory: artifacts


In [4]:
import pandas as pd

DATA_PATH = "cardio_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (68742, 13)


,id,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
0,0,2,168,62.0,110,80,1,1,0,0,1,0,50.357290
1,1,1,156,85.0,140,90,3,1,0,0,1,1,55.381246
2,2,1,165,64.0,130,70,3,1,0,0,0,1,51.627652
3,3,2,169,82.0,150,100,1,1,0,0,1,1,48.249144
4,4,1,156,56.0,100,60,1,1,0,0,0,0,47.841205


In [5]:
# Display dataset columns
print("Dataset columns:")
print(df.columns.tolist())

# Define target
TARGET = "cardio"

# Define input features
FEATURES = [col for col in df.columns if col != TARGET]

print("\nTarget:", TARGET)
print("Number of input features:", len(FEATURES))
print("Input features:", FEATURES)

Dataset columns:
['id', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio', 'age_years']

Target: cardio
Number of input features: 12
Input features: ['id', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'age_years']


In [6]:
# Original features used by the model
BASE_FEATURES = [
    "gender",
    "height",
    "weight",
    "ap_hi",
    "ap_lo",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active",
    "age_years"
]

# Engineered features
ENGINEERED_FEATURES = [
    "bmi",
    "pulse_pressure",
    "map"
]

# Final model input features
MODEL_FEATURES = BASE_FEATURES + ENGINEERED_FEATURES

print("Number of base features:", len(BASE_FEATURES))
print("Number of engineered features:", len(ENGINEERED_FEATURES))
print("Total model features:", len(MODEL_FEATURES))

print("\nModel features:")
print(MODEL_FEATURES)

Number of base features: 11
Number of engineered features: 3
Total model features: 14

Model features:
['gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'age_years', 'bmi', 'pulse_pressure', 'map']


##  Reproduce the Training Preprocessing

To ensure consistency with the final trained model, the preprocessing steps used during training are reproduced here.

Records with inconsistent blood pressure values (`ap_hi <= ap_lo`) are removed before feature engineering.

The final feature-engineered dataset is then split into training, validation, and test sets using the same random state and stratification strategy used during model development.

In [7]:
# Reproduce the cleaning step used during training

df_clean = df[df["ap_hi"] > df["ap_lo"]].copy()

print("Shape after removing inconsistent BP records:", df_clean.shape)

Shape after removing inconsistent BP records: (68639, 13)


In [8]:
# Recreate the engineered features

df_clean["bmi"] = df_clean["weight"] / ((df_clean["height"] / 100) ** 2)

df_clean["pulse_pressure"] = df_clean["ap_hi"] - df_clean["ap_lo"]

df_clean["map"] = df_clean["ap_lo"] + (df_clean["pulse_pressure"] / 3)

print("Feature-engineered shape:", df_clean.shape)

df_clean[MODEL_FEATURES].head()

Feature-engineered shape: (68639, 16)


,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,age_years,bmi,pulse_pressure,map
0,2,168,62.0,110,80,1,1,0,0,1,50.357290,21.967120,30,90.000000
1,1,156,85.0,140,90,3,1,0,0,1,55.381246,34.927679,50,106.666667
2,1,165,64.0,130,70,3,1,0,0,0,51.627652,23.507805,60,90.000000
3,2,169,82.0,150,100,1,1,0,0,1,48.249144,28.710479,50,116.666667
4,1,156,56.0,100,60,1,1,0,0,0,47.841205,23.011177,40,73.333333


##  Split the Data

The feature-engineered dataset is divided into training, validation, and test sets.

The same split strategy used during model development is maintained to ensure consistency with the final trained model.

Stratification is used to preserve the distribution of the target classes across the three sets.

In [9]:
from sklearn.model_selection import train_test_split

# Select model features and target
X = df_clean[MODEL_FEATURES].copy()
y = df_clean[TARGET].copy()

# First split: training and temporary set
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

# Second split: validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Training shape: (48047, 14)
Validation shape: (10296, 14)
Test shape: (10296, 14)


##  Scale the Features

The input features are standardized using `StandardScaler`.

The scaler is fitted only on the training data and then applied to the validation and test sets.

This prevents data leakage and ensures that the same preprocessing transformation can later be saved and reused during deployment.

In [10]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaler to validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training scaled shape:", X_train_scaled.shape)
print("Validation scaled shape:", X_val_scaled.shape)
print("Test scaled shape:", X_test_scaled.shape)

Training scaled shape: (48047, 14)
Validation scaled shape: (10296, 14)
Test scaled shape: (10296, 14)


##  Load the Serialized Final Model

The trained final neural network is loaded from the saved Keras model artifact.

The model is loaded without retraining to preserve the final model selected during development.

In [11]:
from tensorflow.keras.models import load_model

# Load the serialized final model
final_model = load_model("final_neural_network.keras")

# Load the selected classification threshold
with open("best_threshold.txt", "r") as f:
    final_threshold = float(f.read())

print("Final model loaded successfully.")
print(f"Final threshold: {final_threshold:.2f}")

final_model.summary()

Final model loaded successfully.
Final threshold: 0.42


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_28 (Dense)                │ (None, 64)             │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,221 (36.02 KB)

 Trainable params: 3,073 (12.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,148 (24.02 KB)

##  Serialize the Preprocessing Scaler

The fitted `StandardScaler` is saved as a separate artifact.

The scaler was fitted only on the training data and will be reused during inference to ensure that new inputs receive the same preprocessing transformation as the training data.

In [12]:
import joblib

# Save the fitted scaler
joblib.dump(scaler, "standard_scaler.joblib")

print("Scaler saved successfully.")

Scaler saved successfully.


## Verify Serialized Artifacts

The serialized model, scaler, and classification threshold are reloaded to verify that they can be used independently after saving.

This confirms that the artifacts are ready for later inference and deployment.

In [13]:
import joblib
from tensorflow.keras.models import load_model

# Reload the serialized artifacts
loaded_model = load_model("final_neural_network.keras")
loaded_scaler = joblib.load("standard_scaler.joblib")

with open("best_threshold.txt", "r") as f:
    loaded_threshold = float(f.read())

print("All serialized artifacts loaded successfully.")
print(f"Threshold: {loaded_threshold:.2f}")
print(f"Scaler features: {loaded_scaler.n_features_in_}")

All serialized artifacts loaded successfully.
Threshold: 0.42
Scaler features: 14


##  Test Inference with Serialized Artifacts

A sample from the test set is passed through the serialized scaler and model.

The saved classification threshold is then used to convert predicted probabilities into binary predictions.

This verifies that the serialized artifacts work correctly together in an inference workflow.

In [14]:
# Select a small sample from the test set
X_sample = X_test.iloc[:10].copy()

# Scale using the reloaded scaler
X_sample_scaled = loaded_scaler.transform(X_sample)

# Predict probabilities using the reloaded model
sample_probabilities = loaded_model.predict(X_sample_scaled, verbose=0).ravel()

# Convert probabilities to class predictions using the saved threshold
sample_predictions = (sample_probabilities >= loaded_threshold).astype(int)

print("Inference test completed successfully.")
print("\nPredicted probabilities:")
print(sample_probabilities)

print("\nPredicted classes:")
print(sample_predictions)

Inference test completed successfully.

Predicted probabilities:
[0.3111349  0.49948144 0.7408931  0.18062626 0.12643942 0.82127565
 0.33017695 0.6464773  0.49392775 0.8281521 ]

Predicted classes:
[0 1 1 0 0 1 0 1 1 1]


##  Validate the Serialized Model on the Test Set

The serialized model and preprocessing scaler are evaluated on the complete test set.

The saved classification threshold is used to generate the final binary predictions.

The resulting metrics are compared with the final test performance obtained during model development.

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Scale the complete test set using the loaded scaler
X_test_scaled_loaded = loaded_scaler.transform(X_test)

# Predict probabilities using the loaded model
test_probabilities = loaded_model.predict(
    X_test_scaled_loaded,
    verbose=0
).ravel()

# Apply the saved threshold
test_predictions = (
    test_probabilities >= loaded_threshold
).astype(int)

# Calculate test metrics
serialized_accuracy = accuracy_score(y_test, test_predictions)
serialized_precision = precision_score(y_test, test_predictions)
serialized_recall = recall_score(y_test, test_predictions)
serialized_f1 = f1_score(y_test, test_predictions)

print("Serialized Model - Test Results")
print("-" * 45)
print(f"Accuracy : {serialized_accuracy:.4f}")
print(f"Precision: {serialized_precision:.4f}")
print(f"Recall   : {serialized_recall:.4f}")
print(f"F1 Score : {serialized_f1:.4f}")
print(f"Threshold: {loaded_threshold:.2f}")

Serialized Model - Test Results
---------------------------------------------
Accuracy : 0.7187
Precision: 0.6931
Recall   : 0.7744
F1 Score : 0.7315
Threshold: 0.42


In [20]:
print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)
print("NumPy:", np.__version__)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
TensorFlow: 2.20.0
Scikit-learn: 1.6.1
Joblib: 1.6.0
NumPy: 2.1.3


In [22]:
# Create a clean, pinned requirements.txt for deployment

requirements = """tensorflow==2.20.0
scikit-learn==1.6.1
joblib==1.6.0
numpy==2.1.3
pandas==2.2.3
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created successfully.")
print(requirements)

requirements.txt created successfully.
tensorflow==2.20.0
scikit-learn==1.6.1
joblib==1.6.0
numpy==2.1.3
pandas==2.2.3



## MLflow Reproducibility Tracking

MLflow is used to record the model configuration and evaluation results so that the deployment artifact can be traced and reproduced.


In [26]:
import mlflow

mlflow.set_experiment("Cardiac_Patient_Monitoring_Sprint_4")

with mlflow.start_run(run_name="Serialized_Final_Model"):

    mlflow.log_param("model_type", "Neural Network")
    mlflow.log_param("architecture", "64-32-1")
    mlflow.log_param("learning_rate", 0.0001)
    mlflow.log_param("dropout", 0.0)
    mlflow.log_param("batch_size", 64)
    mlflow.log_param("threshold", final_threshold)
    mlflow.log_param("seed", 42)

    mlflow.log_metric("test_accuracy", 0.7187)
    mlflow.log_metric("test_precision", 0.6931)
    mlflow.log_metric("test_recall", 0.7744)
    mlflow.log_metric("test_f1", 0.7315)

    print("MLflow run logged successfully.")

2026/09/14 18:29:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/14 18:29:01 INFO mlflow.store.db.utils: Updating database tables
2026/09/14 18:29:05 INFO mlflow.tracking.fluent: Experiment with name 'Cardiac_Patient_Monitoring_Sprint_4' does not exist. Creating a new experiment.


MLflow run logged successfully.


In [27]:
# Verify the complete serialized inference pipeline

X_test_scaled = loaded_scaler.transform(X_test)

test_probabilities = final_model.predict(X_test_scaled, verbose=0).ravel()
test_predictions = (test_probabilities >= final_threshold).astype(int)

print("Serialized inference verification completed.")
print("Test samples:", len(test_predictions))
print("Threshold:", final_threshold)
print("First 10 probabilities:", test_probabilities[:10])
print("First 10 predictions:", test_predictions[:10])

Serialized inference verification completed.
Test samples: 10296
Threshold: 0.4200000000000001
First 10 probabilities: [0.31113487 0.49948144 0.7408931  0.18062621 0.12643942 0.8212757
 0.33017695 0.6464773  0.49392775 0.8281521 ]
First 10 predictions: [0 1 1 0 0 1 0 1 1 1]


## Conclusion

In this sprint, the final cardiac patient monitoring model was prepared for deployment. The trained neural network, preprocessing scaler, and optimized classification threshold were successfully serialized and loaded back to reproduce predictions. A pinned `requirements.txt` was also created, random seeds were fixed for reproducibility, and the model configuration and evaluation results were tracked using MLflow.

The serialized inference pipeline was successfully verified on the test set, confirming that the saved artifacts can be reused consistently in the deployment environment.
